# 👑 System One — Milestone 2 & 3 (Colab A100): Two-Stage MoS Residual Scar Experts (`Qwen3-Embedding-8B` 8-Depth)
Este notebook executa o treinamento fim-a-fim em **2 Estágios** dos nossos **Especialistas de Cicatriz Residuais MoS** (`rank=16`, zero LoRAs de terceiros) acoplados à extração simultânea de **8 Profundidades (`[4, 8, 12, 16, 20, 24, 30, 36]`)** e **Janelas Deslizantes Sem Truncamento (`max_len=512, stride=384`)**:
1. **Estágio 1:** Adaptação geral no corpus limpo desduplicado (`pool_v4.parquet` + `passages.csv` proxy-matched, sem sobreposição com `train.csv`).
2. **Estágio 2:** Calibração fina no registro exato do host (`train.csv`, 5-Fold `StratifiedGroupKFold` por `prompt_id`), exportando os 5 estados `.safetensors` e o backbone NF4 MoS diretamente para o Kaggle.

In [ ]:
!pip install -q -U bitsandbytes accelerate safetensors transformers kaggle pyarrow scikit-learn
import torch
print('GPU:', torch.cuda.get_device_name(0), '| BF16:', torch.cuda.is_bf16_supported())


### Etapa 1: Autenticação Kaggle & Download dos Corpora Oficiais e Estendidos Limpos
Faça upload do seu `kaggle.json` (ou defina no Colab Secrets) para baixar `train.csv`, `pool_v4.parquet` e `passages.csv` e publicar automaticamente os artefatos ao final.

In [ ]:
import os, pathlib
from google.colab import files
kaggle_dir = pathlib.Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
if not (kaggle_dir / 'kaggle.json').exists():
    uploaded = files.upload()
    for k, v in uploaded.items():
        (kaggle_dir / 'kaggle.json').write_bytes(v)
os.chmod(kaggle_dir / 'kaggle.json', 0o600)

!mkdir -p /content/ntl_data
!kaggle competitions download -c name-that-llm -f train.csv -p /content/ntl_data -q && unzip -o -q /content/ntl_data/train.csv.zip -d /content/ntl_data || true
!kaggle datasets download -d nishantdahal/ntl-handwriting-blend -f pool_v4.parquet -p /content/ntl_data --unzip -q
!kaggle datasets download -d prvsiyan/prvsiyans-extended-name-that-llm-dataset -f passages.csv -p /content/ntl_data --unzip -q
!ls -lh /content/ntl_data


### Etapa 2: Treinamento 2-Estágios dos Especialistas de Cicatriz MoS + 8-Depth Readout + `SCache O(1)`
Grava `train_two_stage_mos_scar_experts_colab.py` e executa o pipeline completo em BF16/NF4 no A100 (~45–55 min).

In [ ]:
from pathlib import Path
Path('/content/train_two_stage_mos_scar_experts_colab.py').write_text('#!/usr/bin/env python3\n"""\n👑 SYSTEM ONE — MILESTONE 2 & 3 (KAGGLE 2xT4 / COLAB A100 / LOCAL VALIDATION):\nTWO-STAGE PROPRIETARY MoS RESIDUAL SCAR EXPERTS + 8-DEPTH MULTI-SLICE ENGINE\n============================================================================\nAlvo de Leaderboard Público: >= 0.740 - 0.755+ (Top 10 em `name-that-llm`).\n100% Arquitetura Proprietária System One (`AGENTS.md` & `SKILL.md`):\n- ZERO adaptadores LoRA comunitários / de terceiros.\n- Especialistas de Cicatriz Residuais MoS (`dW_scar = scale * B @ A`, rank=16) em FP32 master\n  injetados diretamente em `q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj`,\n  exportados no formato `.scar_A` / `.scar_B` nativamente suportado pelo `extract_mos_nf4_streaming`!\n- Suporta execução universal sem dependências frágeis:\n  * Kaggle `2x T4` (offline, sem bitsandbytes): Pipeline Parallel `device_map="auto"` em `float16`\n    com parâmetros Scar + Head em `float32` puro (zero overflow/underflow em SM 7.5!).\n  * Colab `A100` (`--quant nf4` ou `--quant bf16`): NF4 4-bit ou BF16 nativo.\n  * Local `CPU/MPS` (`--quant fp32` / `fp16`): validação determinística fim-a-fim e teste de paridade!\n- Centralização Bessel `sqrt(m / (m - 1))` + Multiplicity Gate `w_twin(m) = 0.55 * (m - 1) / (m + 2)`\n  + Amostragem Multi-Multiplicidade `m in {1, 2, 3, 6, 16}` no Estágio 2 (imune a grupos pequenos!).\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport gc\nimport json\nimport math\nimport os\nimport time\nfrom collections import defaultdict\nfrom contextlib import nullcontext\nfrom pathlib import Path\nfrom typing import Dict, List, Sequence, Tuple\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom safetensors.torch import save_file\nfrom transformers import AutoModel, AutoTokenizer\n\nLABELS = [\n    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",\n    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",\n    "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5", "claude-fable-5",\n    "qwen3.8-max", "glm-5.3", "glm-5.3-flash", "kimi-k3",\n]\nL2I = {l: i for i, l in enumerate(LABELS)}\nK = len(LABELS)\n\nCAPTURE_LAYERS_QWEN3_8B = [4, 8, 12, 16, 20, 24, 30, 36]\nSCAR_TARGET_MODULES = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")\n\n\ndef resolve_capture_layers(n_layers: int, spec: str = "auto") -> List[int]:\n    if spec != "auto":\n        vals = [int(x.strip()) for x in spec.split(",") if x.strip()]\n        return [v for v in vals if 1 <= v <= n_layers]\n    if n_layers == 36:\n        return list(CAPTURE_LAYERS_QWEN3_8B)\n    k = min(8, n_layers)\n    layers = sorted(set(max(1, int(round(i * n_layers / k))) for i in range(1, k + 1)))\n    return layers\n\n\n# ==============================================================================\n# 1. Camada Proprietária MoS Residual Scar Expert (FP32 Master Precision)\n# ==============================================================================\nclass MoSResidualScarLinear(nn.Module):\n    """\n    Envolve uma projeção base congelada (NF4, FP16 ou BF16, em qualquer GPU do `device_map`)\n    com um Especialista de Cicatriz Residual de posto baixo mantido em FP32 Master:\n      `y = base(x) + (alpha / r) * (drop(x.float()) @ A.T @ B.T)`\n    Zero underflow em T4 (SM 7.5) sem necessidade de GradScaler, e fusão exata\n    `w_out.add_((sB @ sA) * 1.0)` em `extract_mos_nf4_streaming`!\n    """\n\n    def __init__(self, base_linear: nn.Module, rank: int = 16, alpha: float = 32.0, dropout: float = 0.05):\n        super().__init__()\n        self.base = base_linear\n        for p in self.base.parameters():\n            p.requires_grad_(False)\n\n        in_features = int(base_linear.in_features)\n        out_features = int(base_linear.out_features)\n        self.rank = rank\n        self.scale = float(alpha) / float(rank)\n        self.drop = nn.Dropout(dropout)\n\n        dev = next(base_linear.parameters()).device\n        self.scar_A = nn.Parameter(torch.empty((rank, in_features), device=dev, dtype=torch.float32))\n        self.scar_B = nn.Parameter(torch.zeros((out_features, rank), device=dev, dtype=torch.float32))\n        nn.init.kaiming_uniform_(self.scar_A, a=math.sqrt(5))\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        base_out = self.base(x)\n        x_f32 = self.drop(x.to(device=self.scar_A.device, dtype=torch.float32))\n        delta = F.linear(F.linear(x_f32, self.scar_A), self.scar_B) * self.scale\n        return base_out + delta.to(device=base_out.device, dtype=base_out.dtype)\n\n\ndef inject_mos_scar_experts(\n    backbone: nn.Module, rank: int = 16, alpha: float = 32.0, dropout: float = 0.05\n) -> Dict[str, MoSResidualScarLinear]:\n    for p in backbone.parameters():\n        p.requires_grad_(False)\n    injected: Dict[str, MoSResidualScarLinear] = {}\n    base_has_model_prefix = hasattr(backbone, "model") and hasattr(backbone.model, "layers")\n    for mod_name, mod in list(backbone.named_modules()):\n        for child_name, child in list(mod.named_children()):\n            if child_name in SCAR_TARGET_MODULES and hasattr(child, "in_features") and hasattr(child, "out_features"):\n                raw_prefix = f"{mod_name}.{child_name}" if mod_name else child_name\n                # Alinha com chaves de `model.safetensors.index.json` (que em Qwen/Llama possuem `model.layers...` ou `layers...`)\n                scar_mod = MoSResidualScarLinear(child, rank=rank, alpha=alpha, dropout=dropout)\n                setattr(mod, child_name, scar_mod)\n                injected[raw_prefix] = scar_mod\n    return injected\n\n\n# ==============================================================================\n# 2. Cabeça Multi-Depth + SCache O(1) + Bessel Multiplicity-Gated Readout\n# ==============================================================================\nclass MultiDepthMoSSCacheClassifier(nn.Module):\n    def __init__(\n        self,\n        backbone: nn.Module,\n        d_model: int,\n        capture_layers_1b: Sequence[int],\n        head_device: str = "cpu",\n        proj_dim: int = 512,\n        scache_dim: int = 32,\n        num_classes: int = 18,\n    ):\n        super().__init__()\n        self.backbone = backbone\n        self.d_model = d_model\n        self.capture_layers_1b = list(capture_layers_1b)\n        self.n_depths = len(self.capture_layers_1b)\n        self.head_device = torch.device(head_device)\n\n        self.depth_norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(self.n_depths)]).to(self.head_device)\n        self.depth_logits = nn.Parameter(torch.zeros(self.n_depths, device=self.head_device))\n        self.proj = nn.Sequential(\n            nn.Linear(self.n_depths * d_model, proj_dim),\n            nn.LayerNorm(proj_dim),\n            nn.SiLU(),\n            nn.Dropout(0.20),\n        ).to(self.head_device)\n        self.scache_key = nn.Linear(proj_dim, scache_dim, bias=False).to(self.head_device)\n        self.scache_scale = nn.Parameter(torch.tensor(0.15, device=self.head_device))\n\n        self.plain_norm = nn.LayerNorm(proj_dim).to(self.head_device)\n        self.plain_head = nn.Linear(proj_dim, num_classes).to(self.head_device)\n\n        self.twin_norm = nn.LayerNorm(proj_dim).to(self.head_device)\n        self.twin_head = nn.Linear(proj_dim, num_classes).to(self.head_device)\n\n    def encode_passages(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n        inp_dev = next(self.backbone.parameters()).device\n        out = self.backbone(\n            input_ids=input_ids.to(inp_dev),\n            attention_mask=attention_mask.to(inp_dev),\n            output_hidden_states=True,\n            use_cache=False,\n        )\n        hs = out.hidden_states\n        m = attention_mask.to(self.head_device).unsqueeze(-1).float()\n        denom = m.sum(dim=1).clamp(min=1.0)\n\n        pooled_slices = []\n        d_weights = F.softmax(self.depth_logits, dim=0) * self.n_depths\n        n_tot_layers = int(self.backbone.config.num_hidden_layers)\n        eps = float(getattr(self.backbone.config, "rms_norm_eps", 1e-6))\n        for idx, l_1b in enumerate(self.capture_layers_1b):\n            h_l = hs[l_1b].to(self.head_device, dtype=torch.float32)\n            if l_1b == n_tot_layers:\n                h_use = h_l\n            else:\n                h_use = h_l * torch.rsqrt(h_l.pow(2).mean(dim=-1, keepdim=True) + eps)\n            p = (h_use * m).sum(dim=1) / denom\n            p_unit = p * torch.rsqrt(p.pow(2).mean(dim=-1, keepdim=True) + 1e-6)\n            p_norm = self.depth_norms[idx](p_unit) * d_weights[idx]\n            pooled_slices.append(p_norm)\n\n        z_cat = torch.cat(pooled_slices, dim=-1)\n        return self.proj(z_cat)\n\n    def apply_scache_and_heads(\n        self, z: torch.Tensor, prompt_ids: torch.Tensor | None = None\n    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        """\n        Retorna `(logits_plain, logits_twin, twin_gate)` onde:\n          - `z_centered` usa correção exata de Bessel `sqrt(m / (m - 1))`.\n          - `twin_gate_i = 0.55 * (m_i - 1) / (m_i - 1 + 3.0)` (`0.0` em singletons `m_i = 1`!).\n        """\n        z = z.to(self.head_device, dtype=torch.float32)\n        z_sc = z\n        z_centered = torch.zeros_like(z)\n        twin_gate = torch.zeros((z.shape[0], 1), device=self.head_device, dtype=torch.float32)\n\n        if prompt_ids is not None:\n            pids = prompt_ids.to(self.head_device)\n            s_proj = self.scache_key(z)\n            z_sc_list = z.clone()\n            for g in torch.unique(pids):\n                mask = pids == g\n                cnt = int(mask.sum().item())\n                if cnt > 1:\n                    bessel = math.sqrt(cnt / float(cnt - 1))\n                    gate_m = (cnt - 1.0) / (cnt - 1.0 + 3.0)\n                    sg = s_proj[mask]\n                    dz = (sg - sg.mean(dim=0, keepdim=True)) * bessel\n                    w_scar = (dz.transpose(0, 1) @ dz) / float(cnt)\n                    res = (dz @ w_scar.transpose(0, 1)) @ self.scache_key.weight\n                    z_sc_list[mask] = z[mask] + (self.scache_scale * gate_m) * res\n                    twin_gate[mask, 0] = 0.55 * gate_m\n            z_sc = z_sc_list\n            for g in torch.unique(pids):\n                mask = pids == g\n                cnt = int(mask.sum().item())\n                if cnt > 1:\n                    bessel = math.sqrt(cnt / float(cnt - 1))\n                    z_centered[mask] = (z_sc[mask] - z_sc[mask].mean(dim=0, keepdim=True)) * bessel\n\n        logits_plain = self.plain_head(self.plain_norm(z_sc))\n        logits_twin = self.twin_head(self.twin_norm(z_centered))\n        return logits_plain, logits_twin, twin_gate\n\n\n# ==============================================================================\n# 3. Corpus Desduplicado 2-Estágios & Amostrador Multi-Multiplicidade\n# ==============================================================================\ndef load_clean_stage1_corpus(train_csv: Path, pool_v4_path: Path, passages_csv: Path):\n    import pandas as pd\n    df_tr = pd.read_csv(train_csv)\n    tr_texts = set(df_tr["text"].fillna("").astype(str).str.strip())\n    tr_pids = set(df_tr["prompt_id"].astype(str))\n\n    frames = []\n    if pool_v4_path.exists():\n        pv4 = pd.read_parquet(pool_v4_path)\n        pv4 = pv4[pv4["model"].isin(L2I)].copy()\n        pv4["text_clean"] = pv4["text"].fillna("").astype(str).str.strip()\n        pv4 = pv4[(~pv4["text_clean"].isin(tr_texts)) & (~pv4["prompt_id"].astype(str).isin(tr_pids))]\n        frames.append(\n            pd.DataFrame({\n                "prompt_id": "pv4_" + pv4["prompt_id"].astype(str),\n                "text": pv4["text"].astype(str),\n                "y": pv4["model"].map(L2I).values,\n                "src_pid": "",\n            })\n        )\n\n    if passages_csv.exists():\n        pa = pd.read_csv(passages_csv)\n        pa = pa[\n            (pa["passes_funnel"] == True)\n            & (pa["is_refusal"] == False)\n            & (pa["is_empty"] == False)\n            & (pa["split"] != "pilot_range")\n            & (pa["split"] != "new")\n            & (pa["model"].isin(L2I))\n        ].copy()\n        pa["text_clean"] = pa["text"].fillna("").astype(str).str.strip()\n        existing = set(frames[0]["text"].str.strip()) if frames else set()\n        pa = pa[(~pa["text_clean"].isin(tr_texts)) & (~pa["text_clean"].isin(existing))]\n        frames.append(\n            pd.DataFrame({\n                "prompt_id": "pa_" + pa["prompt_id"].astype(str),\n                "text": pa["text"].astype(str),\n                "y": pa["model"].map(L2I).values,\n                "src_pid": pa["source_prompt_id"].fillna("").astype(str),\n            })\n        )\n\n    full_ext = pd.concat(frames, ignore_index=True)\n    print(f"✓ Stage-1 Clean Deduplicated Corpus: {len(full_ext)} passages across {full_ext[\'prompt_id\'].nunique()} prompts")\n    return full_ext\n\n\ndef build_prompt_grouped_batches(\n    df, batch_size: int = 16, seed: int = 42, multiscale_m: bool = False\n) -> List[Tuple[np.ndarray, np.ndarray]]:\n    """\n    Retorna lista de `(row_indices, group_ids_in_batch)`.\n    Quando `multiscale_m=True` (Estágio 2), particiona aleatoriamente cada prompt em subgrupos\n    `m in {1, 2, 3, 6, 16}` para calibrar nativamente a cabeça `Twin` em grupos pequenos!\n    """\n    rng = np.random.default_rng(seed)\n    by_prompt: Dict[str, List[int]] = defaultdict(list)\n    for idx, pid in enumerate(df["prompt_id"].astype(str).values):\n        by_prompt[pid].append(idx)\n\n    pids = list(by_prompt.keys())\n    rng.shuffle(pids)\n\n    chunks_rows: List[Tuple[int, int]] = []\n    gid = 1\n    for pid in pids:\n        rows = by_prompt[pid].copy()\n        rng.shuffle(rows)\n        if not multiscale_m:\n            for r in rows:\n                chunks_rows.append((r, gid))\n            gid += 1\n        else:\n            pos = 0\n            while pos < len(rows):\n                m_sub = int(rng.choice([1, 2, 3, 4, 8, 16], p=[0.10, 0.25, 0.25, 0.15, 0.10, 0.15]))\n                seg = rows[pos : pos + m_sub]\n                for r in seg:\n                    chunks_rows.append((r, gid))\n                gid += 1\n                pos += len(seg)\n\n    batches: List[Tuple[np.ndarray, np.ndarray]] = []\n    for s in range(0, len(chunks_rows), batch_size):\n        seg = chunks_rows[s : s + batch_size]\n        if len(seg) >= 2:\n            r_arr = np.array([x[0] for x in seg], dtype=np.int64)\n            g_arr = np.array([x[1] for x in seg], dtype=np.int64)\n            batches.append((r_arr, g_arr))\n    return batches\n\n\n# ==============================================================================\n# 4. Exportador `.safetensors` Compatível com `extract_mos_nf4_streaming`\n# ==============================================================================\ndef detect_safetensors_key_prefix(backbone_dir: Path | None) -> str:\n    """Detecta se `model.safetensors.index.json` ou `model.safetensors` usa prefixo `model.layers.` ou `layers.`."""\n    if backbone_dir is not None:\n        idx_file = backbone_dir / "model.safetensors.index.json"\n        single_file = backbone_dir / "model.safetensors"\n        if idx_file.exists():\n            keys = list(json.loads(idx_file.read_text()).get("weight_map", {}).keys())\n        elif single_file.exists():\n            from safetensors import safe_open\n            with safe_open(str(single_file), framework="pt", device="cpu") as f_st:\n                keys = list(f_st.keys())\n        else:\n            keys = []\n        if any(k.startswith("model.layers.") for k in keys):\n            return "model."\n        if any(k.startswith("layers.") for k in keys):\n            return ""\n    return ""\n\n\ndef export_fold_safetensors(\n    model: MultiDepthMoSSCacheClassifier,\n    injected: Dict[str, MoSResidualScarLinear],\n    out_path: Path,\n    key_prefix: str = "",\n):\n    tensors: Dict[str, torch.Tensor] = {}\n    for raw_prefix, scar_mod in injected.items():\n        # Normaliza prefixo (`layers.0...` vs `model.layers.0...`) conforme o checkpoint no disco!\n        clean_p = raw_prefix[len("model.") :] if raw_prefix.startswith("model.") else raw_prefix\n        final_p = f"{key_prefix}{clean_p}"\n        tensors[f"{final_p}.scar_A"] = (scar_mod.scar_A.detach().float() * scar_mod.scale).to(torch.float16).cpu()\n        tensors[f"{final_p}.scar_B"] = scar_mod.scar_B.detach().float().to(torch.float16).cpu()\n\n    for k, v in model.state_dict().items():\n        if not k.startswith("backbone."):\n            tensors[f"head.{k}"] = v.detach().float().cpu()\n\n    save_file(tensors, str(out_path))\n\n\ndef get_autocast_ctx(device_str: str, quant_mode: str):\n    if not device_str.startswith("cuda") or not torch.cuda.is_available():\n        return nullcontext()\n    cc_major = torch.cuda.get_device_capability(0)[0]\n    if quant_mode in ("bf16", "nf4") and cc_major >= 8:\n        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)\n    return torch.autocast(device_type="cuda", dtype=torch.float16)\n\n\n@torch.no_grad()\ndef evaluate_covering_windows(\n    model: MultiDepthMoSSCacheClassifier,\n    tokenizer,\n    texts: List[str],\n    prompt_groups: np.ndarray,\n    max_len: int = 512,\n    stride: int = 384,\n    batch_size: int = 24,\n    device: str = "cuda:0",\n    quant_mode: str = "fp16",\n) -> np.ndarray:\n    model.eval()\n    enc_all = tokenizer(texts, truncation=False, padding=False)["input_ids"]\n    eos_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0\n\n    win_ids, doc_idx, win_wts = [], [], []\n    for i, seq in enumerate(enc_all):\n        L = len(seq)\n        if L <= max_len:\n            win_ids.append(seq if L > 0 else [eos_id])\n            doc_idx.append(i)\n            win_wts.append(float(max(L, 1)))\n        else:\n            starts = list(range(0, L - max_len + 1, stride))\n            if starts[-1] + max_len < L:\n                starts.append(L - max_len)\n            for st in starts:\n                ch = seq[st : st + max_len]\n                win_ids.append(ch)\n                doc_idx.append(i)\n                win_wts.append(float(len(ch)))\n\n    order = np.argsort([len(s) for s in win_ids], kind="stable")\n    proj_dim = model.plain_head.in_features\n    Z_win = np.zeros((len(win_ids), proj_dim), dtype=np.float32)\n\n    for b_s in range(0, len(win_ids), batch_size):\n        b_i = order[b_s : b_s + batch_size]\n        tok_b = tokenizer.pad({"input_ids": [win_ids[j] for j in b_i]}, padding=True, return_tensors="pt")\n        with get_autocast_ctx(device, quant_mode):\n            z_b = model.encode_passages(tok_b["input_ids"], tok_b["attention_mask"])\n        Z_win[b_i] = z_b.float().cpu().numpy()\n\n    Z_doc = np.zeros((len(texts), proj_dim), dtype=np.float32)\n    w_sum = np.zeros((len(texts), 1), dtype=np.float32)\n    doc_idx_arr = np.asarray(doc_idx, dtype=np.int64)\n    win_wts_arr = np.asarray(win_wts, dtype=np.float32)\n    np.add.at(Z_doc, doc_idx_arr, Z_win * win_wts_arr[:, None])\n    np.add.at(w_sum[:, 0], doc_idx_arr, win_wts_arr)\n    Z_doc /= np.maximum(w_sum, 1e-6)\n\n    z_t = torch.from_numpy(Z_doc)\n    _, u_inv = np.unique(prompt_groups, return_inverse=True)\n    g_t = torch.from_numpy(u_inv)\n    logits_p, logits_c, twin_gate = model.apply_scache_and_heads(z_t, g_t)\n    probs = (1.0 - twin_gate) * F.softmax(logits_p, dim=-1) + twin_gate * F.softmax(logits_c, dim=-1)\n    return probs.cpu().numpy()\n\n\n# ==============================================================================\n# 5. Entrypoint CLI Universal (Kaggle 2xT4 / Colab A100 / Local Smoke Test)\n# ==============================================================================\ndef main():\n    import pandas as pd\n    from sklearn.metrics import balanced_accuracy_score, log_loss\n    from sklearn.model_selection import StratifiedGroupKFold\n\n    parser = argparse.ArgumentParser(description="System One Two-Stage MoS Residual Scar Experts")\n    parser.add_argument("--model-id", default="Qwen/Qwen3-Embedding-8B")\n    parser.add_argument("--train-csv", required=True)\n    parser.add_argument("--pool-v4", required=True)\n    parser.add_argument("--passages-csv", required=True)\n    parser.add_argument("--out-dir", default="./titan4_qwen3_emb_8b_mos_artifacts")\n    parser.add_argument("--quant", choices=["auto", "nf4", "fp16", "bf16", "fp32"], default="auto")\n    parser.add_argument("--save-backbone", action="store_true", help="Salva cópia do backbone quantizado NF4")\n    parser.add_argument("--capture-layers", default="auto")\n    parser.add_argument("--stage1-max-rows", type=int, default=27000)\n    parser.add_argument("--stage1-epochs", type=int, default=1)\n    parser.add_argument("--stage2-epochs", type=int, default=4)\n    parser.add_argument("--n-folds", type=int, default=5)\n    parser.add_argument("--max-folds-run", type=int, default=5)\n    parser.add_argument("--batch-size", type=int, default=16)\n    parser.add_argument("--max-len", type=int, default=512)\n    parser.add_argument("--rank", type=int, default=16)\n    args = parser.parse_args()\n\n    out_dir = Path(args.out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    has_cuda = torch.cuda.is_available()\n    n_gpus = torch.cuda.device_count() if has_cuda else 0\n    head_device = f"cuda:{n_gpus - 1}" if n_gpus > 0 else "cpu"\n\n    quant_mode = args.quant\n    if quant_mode == "auto":\n        if not has_cuda:\n            quant_mode = "fp32"\n        elif torch.cuda.get_device_capability(0)[0] >= 8:\n            try:\n                import bitsandbytes  # noqa: F401\n                quant_mode = "nf4"\n            except Exception:\n                quant_mode = "bf16"\n        else:\n            quant_mode = "fp16"\n\n    print(f"🚀 Launching Two-Stage MoS Scar-Expert Training | GPUs={n_gpus} | Mode={quant_mode} | Model={args.model_id}", flush=True)\n\n    df_train = pd.read_csv(args.train_csv)\n    df_train["text"] = df_train["text"].fillna("").astype(str)\n    df_train["y"] = df_train["model"].map(L2I).values\n    prompts_train = df_train["prompt_id"].astype(str).values\n\n    df_ext = load_clean_stage1_corpus(Path(args.train_csv), Path(args.pool_v4), Path(args.passages_csv))\n    if len(df_ext) > args.stage1_max_rows:\n        per_cls = max(1, args.stage1_max_rows // K)\n        df_ext = (\n            df_ext.groupby("y", group_keys=False)\n            .apply(lambda g: g.sample(n=min(len(g), per_cls), random_state=42))\n            .reset_index(drop=True)\n        )\n        print(f"✓ Balanced Stage-1 Subset: {len(df_ext)} rows (~{per_cls}/class)", flush=True)\n\n    tokenizer = AutoTokenizer.from_pretrained(args.model_id, padding_side="left")\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token = tokenizer.eos_token\n\n    model_src_path = Path(args.model_id) if Path(args.model_id).exists() else None\n    if quant_mode == "nf4":\n        from transformers import BitsAndBytesConfig\n        comp_dt = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16\n        bnb_cfg = BitsAndBytesConfig(\n            load_in_4bit=True,\n            bnb_4bit_quant_type="nf4",\n            bnb_4bit_use_double_quant=True,\n            bnb_4bit_compute_dtype=comp_dt,\n        )\n        base_backbone = AutoModel.from_pretrained(args.model_id, quantization_config=bnb_cfg, device_map="auto")\n    elif quant_mode in ("fp16", "bf16") and has_cuda:\n        dt = torch.float16 if quant_mode == "fp16" else torch.bfloat16\n        base_backbone = AutoModel.from_pretrained(args.model_id, dtype=dt, device_map="auto")\n    else:\n        base_backbone = AutoModel.from_pretrained(args.model_id, dtype=torch.float32).to("cpu")\n\n    base_backbone.config.use_cache = False\n    if has_cuda and hasattr(base_backbone, "gradient_checkpointing_enable"):\n        base_backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n        if hasattr(base_backbone, "enable_input_require_grads"):\n            base_backbone.enable_input_require_grads()\n\n    bb_save_dir = out_dir / "backbone_nf4_mos"\n    if args.save_backbone and not (bb_save_dir / "config.json").exists():\n        print(f"💾 Exportando backbone MoS para {bb_save_dir}...", flush=True)\n        base_backbone.save_pretrained(bb_save_dir, safe_serialization=True, max_shard_size="4GB")\n        tokenizer.save_pretrained(bb_save_dir)\n        model_src_path = bb_save_dir\n\n    key_prefix = detect_safetensors_key_prefix(model_src_path)\n    n_tot_layers = int(base_backbone.config.num_hidden_layers)\n    cap_layers = resolve_capture_layers(n_tot_layers, args.capture_layers)\n    d_model = int(base_backbone.config.hidden_size)\n\n    injected = inject_mos_scar_experts(base_backbone, rank=args.rank, alpha=2.0 * args.rank, dropout=0.05)\n    model = MultiDepthMoSSCacheClassifier(\n        base_backbone, d_model=d_model, capture_layers_1b=cap_layers, head_device=head_device\n    )\n\n    trainable_params = [p for p in model.parameters() if p.requires_grad]\n    print(\n        f"✓ Injected {len(injected)} MoS Scar projections | Capture={cap_layers} | "\n        f"KeyPrefix=\'{key_prefix}\' | Trainable={sum(p.numel() for p in trainable_params):,}",\n        flush=True,\n    )\n\n    def run_epoch(df_ep: pd.DataFrame, opt: torch.optim.Optimizer, lr_sched, seed: int, multiscale: bool) -> float:\n        model.train()\n        batches = build_prompt_grouped_batches(df_ep, batch_size=args.batch_size, seed=seed, multiscale_m=multiscale)\n        tot_loss, n_steps = 0.0, 0\n        texts_all = df_ep["text"].tolist()\n        y_all = df_ep["y"].values\n\n        for b_idx, g_idx in batches:\n            b_texts = [texts_all[i] for i in b_idx]\n            tok = tokenizer(b_texts, truncation=True, max_length=args.max_len, padding=True, return_tensors="pt")\n            yt = torch.from_numpy(y_all[b_idx]).long().to(head_device)\n            gt = torch.from_numpy(g_idx).long().to(head_device)\n\n            opt.zero_grad(set_to_none=True)\n            with get_autocast_ctx(head_device, quant_mode):\n                z = model.encode_passages(tok["input_ids"], tok["attention_mask"])\n            lp, lc, twin_gate = model.apply_scache_and_heads(z.float(), gt)\n            loss_p = F.cross_entropy(lp, yt, label_smoothing=0.05)\n            has_sib = (twin_gate[:, 0] > 1e-5)\n            if bool(has_sib.any()):\n                loss_c = F.cross_entropy(lc[has_sib], yt[has_sib], label_smoothing=0.05)\n                loss = 0.50 * loss_p + 0.50 * loss_c\n            else:\n                loss = loss_p\n\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)\n            opt.step()\n            if lr_sched is not None:\n                lr_sched.step()\n            tot_loss += float(loss.item())\n            n_steps += 1\n            if n_steps % 100 == 0:\n                vram = (torch.cuda.max_memory_allocated(0) / (1024**3)) if has_cuda else 0.0\n                print(f"      step {n_steps}/{len(batches)} | avg_loss={tot_loss/n_steps:.4f} | gpu0_peak={vram:.2f} GiB", flush=True)\n        return tot_loss / max(n_steps, 1)\n\n    # ==========================================================================\n    # ESTÁGIO 1: Pré-Adaptação Forense no Open Pool Limpo\n    # ==========================================================================\n    df_ext_stage1 = df_ext[df_ext["src_pid"] == ""].reset_index(drop=True)\n    if args.stage1_epochs > 0 and len(df_ext_stage1) >= 4:\n        print("\\n=== [STAGE 1] Pre-Adapting MoS Residual Scar Experts on Clean Open Pool ===", flush=True)\n        total_s1_steps = max(len(build_prompt_grouped_batches(df_ext_stage1, args.batch_size)) * args.stage1_epochs, 1)\n        opt_s1 = torch.optim.AdamW(trainable_params, lr=2e-4, weight_decay=0.02)\n        sched_s1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_s1, T_max=total_s1_steps, eta_min=2e-5)\n        for ep in range(args.stage1_epochs):\n            t0 = time.time()\n            l_ep = run_epoch(df_ext_stage1, opt_s1, sched_s1, seed=100 + ep, multiscale=False)\n            print(f"  Stage-1 Epoch {ep+1}/{args.stage1_epochs} | Loss={l_ep:.4f} | Time={time.time()-t0:.1f}s", flush=True)\n\n    stage1_snapshot = {\n        k: v.detach().cpu().clone()\n        for k, v in model.state_dict().items()\n        if any(t in k for t in ("scar_A", "scar_B")) or not k.startswith("backbone.")\n    }\n\n    # ==========================================================================\n    # ESTÁGIO 2: Calibração Fina Multi-Multiplicidade no Registro do Host\n    # ==========================================================================\n    print("\\n=== [STAGE 2] Multi-Multiplicity Host Calibration on Official `train.csv` ===", flush=True)\n    sgkf = StratifiedGroupKFold(n_splits=args.n_folds)\n    oof_probs = np.zeros((len(df_train), K), dtype=np.float32)\n    evaluated_mask = np.zeros(len(df_train), dtype=bool)\n\n    for fold_i, (tr_idx, va_idx) in enumerate(sgkf.split(df_train, df_train["y"].values, groups=prompts_train)):\n        if fold_i >= args.max_folds_run:\n            break\n        tf0 = time.time()\n        cur_sd = model.state_dict()\n        for k, v in stage1_snapshot.items():\n            cur_sd[k].copy_(v.to(cur_sd[k].device))\n\n        df_fold_tr = df_train.iloc[tr_idx][["prompt_id", "text", "y"]].reset_index(drop=True)\n        est_batches = build_prompt_grouped_batches(df_fold_tr, args.batch_size, seed=500 + fold_i, multiscale_m=True)\n        total_s2_steps = max(len(est_batches) * args.stage2_epochs, 1)\n        opt_s2 = torch.optim.AdamW(trainable_params, lr=8e-5, weight_decay=0.03)\n        sched_s2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_s2, T_max=total_s2_steps, eta_min=5e-6)\n\n        for ep in range(args.stage2_epochs):\n            l_s2 = run_epoch(df_fold_tr, opt_s2, sched_s2, seed=500 + fold_i * 10 + ep, multiscale=True)\n            print(f"    Fold {fold_i+1} Epoch {ep+1}/{args.stage2_epochs} | Loss={l_s2:.4f}", flush=True)\n\n        va_texts = df_train.iloc[va_idx]["text"].tolist()\n        va_groups = prompts_train[va_idx]\n        p_va = evaluate_covering_windows(\n            model, tokenizer, va_texts, va_groups, max_len=args.max_len, device=head_device, quant_mode=quant_mode\n        )\n        oof_probs[va_idx] = p_va\n        evaluated_mask[va_idx] = True\n\n        fold_acc = balanced_accuracy_score(df_train["y"].values[va_idx], p_va.argmax(axis=1))\n        fold_path = out_dir / f"titan4_qwen3_emb_8b_fold{fold_i}_scars.safetensors"\n        export_fold_safetensors(model, injected, fold_path, key_prefix=key_prefix)\n        print(\n            f"  ✓ Fold {fold_i+1}/{args.n_folds} | Host Val BalAcc = {fold_acc*100:.2f}% | "\n            f"Saved {fold_path.name} ({time.time()-tf0:.1f}s)",\n            flush=True,\n        )\n        gc.collect()\n        if has_cuda:\n            torch.cuda.empty_cache()\n\n    eval_y = df_train["y"].values[evaluated_mask]\n    eval_p = oof_probs[evaluated_mask]\n    oof_bal = balanced_accuracy_score(eval_y, eval_p.argmax(axis=1))\n    oof_ll = log_loss(eval_y, np.clip(eval_p, 1e-9, 1.0), labels=list(range(K)))\n    np.save(out_dir / "oof_titan4_qwen3_emb_8b_8depth.npy", oof_probs)\n    meta = {\n        "schema": "system-one-titan4-mos-scars-v2",\n        "model_id": args.model_id,\n        "capture_layers_1b": cap_layers,\n        "key_prefix": key_prefix,\n        "rank": args.rank,\n        "n_folds_completed": int(min(args.n_folds, args.max_folds_run)),\n        "oof_balanced_accuracy": float(oof_bal),\n        "oof_log_loss": float(oof_ll),\n        "labels": LABELS,\n    }\n    (out_dir / "meta_titan4_qwen3_emb_8b.json").write_text(json.dumps(meta, indent=2))\n    print("=" * 84, flush=True)\n    print(f"🏆 TITAN 4 OOF BALANCED ACC: {oof_bal*100:.2f}% | LL: {oof_ll:.4f}", flush=True)\n    print("=" * 84, flush=True)\n\n\nif __name__ == "__main__":\n    main()\n')
!python3 /content/train_two_stage_mos_scar_experts_colab.py \
  --model-id Qwen/Qwen3-Embedding-8B \
  --train-csv /content/ntl_data/train.csv \
  --pool-v4 /content/ntl_data/pool_v4.parquet \
  --passages-csv /content/ntl_data/passages.csv \
  --out-dir /content/titan4_qwen3_emb_8b_mos_artifacts \
  --stage1-max-rows 36000 --stage1-epochs 2 --stage2-epochs 4 --batch-size 16 --rank 16


### Etapa 3: Publicação Automática dos Datasets Privados no Kaggle (`Backbone NF4` + `5-Fold Scar Experts`)
Publica `thetod/name-that-llm-titan4-qwen3emb-mos-backbone` e `thetod/name-that-llm-titan4-qwen3emb-mos-scars` prontos para acoplamento imediato no kernel `thetod/mixture-of-scars-70b-mos-scache-name-that-llm`.

In [ ]:
import json, shutil
from pathlib import Path

art = Path('/content/titan4_qwen3_emb_8b_mos_artifacts')
bb_dir = art / 'backbone_nf4_mos'
(bb_dir / 'dataset-metadata.json').write_text(json.dumps({
    'title': 'Name That LLM Titan 4 Qwen3 Emb MoS Backbone',
    'id': 'thetod/name-that-llm-titan4-qwen3emb-mos-backbone',
    'licenses': [{'name': 'CC0-1.0'}]
}, indent=2))

scars_dir = Path('/content/titan4_scars_upload')
scars_dir.mkdir(exist_ok=True)
for f in art.glob('*.safetensors'): shutil.copy2(f, scars_dir / f.name)
for f in art.glob('*.json'): shutil.copy2(f, scars_dir / f.name)
for f in art.glob('*.npy'): shutil.copy2(f, scars_dir / f.name)
(scars_dir / 'dataset-metadata.json').write_text(json.dumps({
    'title': 'Name That LLM Titan 4 Qwen3 Emb MoS Scars',
    'id': 'thetod/name-that-llm-titan4-qwen3emb-mos-scars',
    'licenses': [{'name': 'CC0-1.0'}]
}, indent=2))

!kaggle datasets create -p /content/titan4_qwen3_emb_8b_mos_artifacts/backbone_nf4_mos --dir-mode zip || kaggle datasets version -p /content/titan4_qwen3_emb_8b_mos_artifacts/backbone_nf4_mos -m 'Update Titan 4 NF4 MoS Backbone' --dir-mode zip
!kaggle datasets create -p /content/titan4_scars_upload --dir-mode zip || kaggle datasets version -p /content/titan4_scars_upload -m 'Update Titan 4 5-Fold MoS Scar Experts' --dir-mode zip


### Conclusão
Após concluir o upload acima, basta anexar `thetod/name-that-llm-titan4-qwen3emb-mos-backbone` e `thetod/name-that-llm-titan4-qwen3emb-mos-scars` ao nosso kernel System One Apex — o motor `extract_mos_nf4_streaming` já detecta e funde automaticamente as 8 profundidades e os tensores `.scar_A`/`.scar_B` em `< 2.5 GiB` de VRAM!